# Adversarial contrastive learning — training the SupCon model against attacks

Contrastive training alone is not a defence until it survives attacks; this notebook trains the SupCon model **adversarially** — the attack (PGD-min2, mimicry) is inside the training loop — in several formulations: adversarial training around the Wrapper model, the joint objective (adversarial loss + SupCon loss together), and end-to-end training of SupConNet. Attack evaluations after each round show what the training bought.

**Where this fits:** this section tests a different defence idea — train the detector with **supervised contrastive learning** so its feature space separates the classes more cleanly, then attack the result with the machinery of `03-attacks/`. The CW machinery aimed at these models continues the lineage of `03-attacks/cw_sine_lineage.ipynb`.

*Consolidated from `adversarial_contrast_learning3.ipynb` (the spine), `adversarial_contrast_learning2.ipynb` and `adversarial_contrast_learning5.ipynb` (originally in `mal_adv5`). The three versions repeat the same train/attack cycles; each formulation is shown once. Outputs cleared.*

## Step 1 — Setup, and the mask arithmetic behind the SupCon loss

# Load dataset and libraries

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader,WeightedRandomSampler
from sklearn.model_selection import train_test_split
from scipy import sparse
import gdown
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,balanced_accuracy_score
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, roc_curve, auc
import time
import os
import math
from sklearn.manifold import TSNE
import seaborn as sns
import pandas as pd
import tqdm
import pickle

torch.manual_seed(0)
np.random.seed(0)

# Device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
download_links = ['https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_2.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y0.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y1.npz',
                  'https://github.com/mostafa-ja/mal_adv3/raw/main/drebin/sparse_matrix_y2.npz',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/insertion_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/removal_array.pkl',
                  'https://github.com/mostafa-ja/mal_adv4/raw/main/dataset/adverserial_attacks_functions.py'
]

output_filepath = '/content/'
for link in download_links:
  gdown.download(link, output_filepath)

In [ ]:
from adverserial_attacks_functions import *

In [ ]:
# Open the .pkl file
with open('/content/insertion_array.pkl', 'rb') as f:
    # Load the object
    insertion_array = pickle.load(f)

# Close the file
f.close()

insertion_array = torch.tensor(insertion_array, dtype=torch.uint8).to(device)
print(len(insertion_array))

# Open the .pkl file
with open('/content/removal_array.pkl', 'rb') as f:
    # Load the object
    removal_array = pickle.load(f)

# Close the file
f.close()

removal_array = torch.tensor(removal_array, dtype=torch.uint8).to(device)
print(len(removal_array))

In [ ]:
# Load dataset
X_train = sparse.load_npz("/content/sparse_matrix_0.npz").toarray()
X_val = sparse.load_npz("/content/sparse_matrix_1.npz").toarray()
X_test = sparse.load_npz("/content/sparse_matrix_2.npz").toarray()

X_train = torch.tensor(X_train, dtype=torch.int8)
X_val = torch.tensor(X_val, dtype=torch.int8)
X_test = torch.tensor(X_test, dtype=torch.int8)


y_train = sparse.load_npz("/content/sparse_matrix_y0.npz").toarray().reshape((-1, 1))
y_val = sparse.load_npz("/content/sparse_matrix_y1.npz").toarray().reshape((-1, 1))
y_test = sparse.load_npz("/content/sparse_matrix_y2.npz").toarray().reshape((-1, 1))

y_train = torch.tensor(y_train, dtype=torch.int8)
y_val = torch.tensor(y_val, dtype=torch.int8)
y_test = torch.tensor(y_test, dtype=torch.int8)


print("Shapes:")
print("x_train:", X_train.shape)
print("x_val:", X_val.shape)
print("x_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

In [ ]:
# number of benigns and malicious sample in training dataset
n_ben = (y_train.squeeze()== 0).sum().item()
n_mal = (y_train.squeeze()== 1).sum().item()

# Combine features and labels into datasets
# we use dtype=torch.int8, for Memory-Efficient here, later we will convert to float
train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)
test_dataset = TensorDataset(X_test, y_test)

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 4
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
for x, labels in train_loader:
    y = labels.squeeze()
    print(y.shape)
    break

In [ ]:
pos_mask = torch.matmul(labels, labels.T).fill_diagonal_(0)
pos_mask

In [ ]:
pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
hard_mask = torch.matmul(labels, preds.T) # [2b, 2b], mask for tile(y_j) == y_i
neg_mask = torch.multiply(inv_mask, hard_mask)
neg_mask

In [ ]:
batch_size = 2
  masks = F.one_hot(torch.arange(2*batch_size), num_classes=2*batch_size)
  masks

In [ ]:
import torch
import torch.nn.functional as F

# Input tensor
z = torch.tensor([[2.0, 1.0, 1.0], [-120.0, 1.0, 1.2]])

# Compute log-softmax along dim=1
log_softmax_z = F.log_softmax(z, dim=1)
print(log_softmax_z)

# SupContrast Model (Encoder + Projector)

In [ ]:
def extract_features(model, dataloader):
    model.cpu()
    model.eval()
    features = []
    labels = []

    with torch.no_grad():
        for inputs, targets in dataloader:
            output, z = model.CoreModel(inputs.to(torch.float32), return_z=True)
            features.append(z.cpu().numpy())
            labels.append(targets.cpu().numpy())

    features = np.concatenate(features, axis=0)
    labels = np.concatenate(labels, axis=0)

    return features, labels


def apply_tsne(features):
    tsne = TSNE(n_components=2, random_state=42)
    tsne_results = tsne.fit_transform(features)
    return tsne_results

def plot_tsne_and_density(tsne_results, labels):
    df = pd.DataFrame({
        'tsne_1': tsne_results[:, 0],
        'tsne_2': tsne_results[:, 1],
        'label': labels.squeeze()
    })

    # Create subplots
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 14))

    # t-SNE scatter plot
    sns.scatterplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, ax=ax1, legend='full')
    ax1.set_title('t-SNE Scatter Plot')

    # Density plot with transparency and contours
    sns.kdeplot(x='tsne_1', y='tsne_2', hue='label', palette='viridis', data=df, fill=True, alpha=0.70, ax=ax2)
    ax2.set_title('Density Plot')

    plt.show()

In [ ]:
def adv_predict(test_loader, model, attack, device, **kwargs):

    model.eval()
    n_samples = 0
    cor_test = 0
    cor_ad_test = 0

    with torch.no_grad():
        for x_test, y_test in test_loader:
            x_test, y_test = x_test.to(torch.float32).to(device), y_test.to(device)
            #outputs = model(x_test)
            #predicted = outputs.argmax(1).unsqueeze(1)
            #acc_test = (predicted == y_test).sum().item() / len(y_test)
            #avg_acc_test.append(acc_test)

            mal_x_batch, mal_y_batch = x_test[y_test.squeeze() == 1], y_test[y_test.squeeze() == 1]
            n_samples += len(mal_y_batch)

            outputs = model(mal_x_batch)
            predicted = outputs.argmax(1)
            cor_test += (predicted == 1).sum().item()

            # Generate adversarial examples for test set
            with torch.enable_grad():
                pertb_mal_x= attack(mal_x_batch, mal_y_batch, model, **kwargs)

            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_test += (y_pred == 1).sum().item()

    # Calculate test accuracy
    print(f"Accuracy of just malwares (without attack): {(cor_test / n_samples) * 100:.4}% | Under attack: {(cor_ad_test / n_samples) * 100:.4}%.")

## Step 2 — The model code and the adversarial-training variants

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(4, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
import torch
from torch._C import device
import torch.nn.functional as F

def distance(x, y, dist, pairwise=False):
    """
        Pairwise distance
        Args:
            x, y: input pair
            dist: distance type ["l2", "l1", "linf", "cosine"]
            pairwise:
        Note:
            Need renormalize if using l1
    """

    assert(len(x.shape) == 2)
    b, d = x.shape

    if pairwise:
        x = torch.unsqueeze(x, dim=0) # [1, b, d]
        y = torch.unsqueeze(y, dim=1) # [b, 1, d]

    if dist == 'l2':
        loss = torch.sqrt(torch.sum(torch.square(x - y), dim=-1, keepdim=False)) # [b,b]

    elif dist == 'l1':
        loss = torch.sum(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'linf':
        loss, _ = torch.max(torch.abs(x - y), dim=-1, keepdim=False) # [b,b]

    elif dist == 'lkinf':
        k = 10
        t, _ = torch.topk(torch.abs(x - y), k=k, dim=-1)
        loss = torch.sum(t, dim=-1, keepdim=False)
        loss /= k

    elif dist == 'cosine':
        # Note that: cosine_similarity range [-1,1]
        # dist=-1 <--> very similar
        # dist=0 <--> orthogonal
        # dist=1 <--> very disimilar
        loss = 0 - torch.cosine_similarity(x, y, dim=-1)

    elif dist == 'matmul':
        x = torch.squeeze(x) # [b,d]
        y = torch.squeeze(y) # [b,d]
        loss = - torch.matmul(x, y.T)

    # loss = loss / torch.max(loss).detach() # normalizing to avoid numerical problem

    if pairwise:
        assert(loss.shape[0] == b)
        assert(loss.shape[1] == b)
    else:
        assert(loss.shape[0] == x.shape[0]) # [b, ]

    return loss

def get_positive_mask(labels):
    # ATTENTION HERE, positive mask will ignore the diagonal

    # l = torch.matmul(labels, labels.T) # [2b,2b]
    # m = torch.ones_like(l).fill_diagonal_(0) # [2b,2b]
    # pos_mask = torch.multiply(l, m) # [2b,2b]

    l = torch.matmul(labels, labels.T).fill_diagonal_(0) # [2b,2b]
    return l

def get_negative_mask(labels):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    return inv_mask

def get_hard_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    hard_mask = torch.matmul(labels, preds.T) # [2b, 2b], mask for tile(y_j) == y_i
    neg_mask = torch.multiply(inv_mask, hard_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask


def get_soft_negative_mask(labels, preds):
    pos_mask = torch.matmul(labels, labels.T) # [2b, 2b], mask for y_j == y_i
    inv_mask = torch.ones_like(pos_mask) - pos_mask # [2b, 2b], mask for y_j != y_i
    soft_mask = torch.matmul(preds, preds.T) # [2b, 2b], mask for tile(y_j) == tile(y_i)
    neg_mask = torch.multiply(inv_mask, soft_mask) # [2b, 2b], mask for (y_j!= y_i & tile(y_j) == y_i)
    return neg_mask

LARGE_NUM = 1e9

def soft_lcscl(hidden,
             labels,
             preds,
             dist,
             neg_type,
             hidden_norm=True,
             temperature=0.07):
    """Compute Local Supervised Contrastive Loss.
    Args:
        hidden: hidden vector (`Tensor`) of shape (2 * bsz, dim).
        labels: labels of hidden vector of shape (2*bsz, classes)
        preds: current prediction
        hidden_norm: whether or not to use normalization on the hidden vector.
        temperature: a `floating` number for temperature scaling.
        dist: distance metric in latent space {'l1', 'l2', 'cosine', 'linf'}
        neg_type:
            'hard': negative set when y_j ~= y_i, but tilde(y_j) == y_i
            'soft': negative set when y_j ~= y_i, but tilde(y_j) == tilde(y_i)
            'merge': negative set when y_i ~= y_i but tilde(y_j) in [y_i,  tilde(y_i)]
            'leaking':
    Returns:
        A loss scalar.

    the choice of temperature:
        temperature = 0.5 based on simCLR paper
        temperature = 0.07 based on SCL paper (they claim that smaller temperature
        benefit training more than higher ones)
    """
    if dist in ['matmul', 'l1', 'l2']:
        assert(hidden_norm is True)
    elif dist in ['cosine']:
        assert(hidden_norm is False)

    # Get (normalized) hidden1 and hidden2.
    if hidden_norm:
        # gradient will pass through the normalization step
        hidden = F.normalize(hidden, p=2, dim=-1)


    batch_size = hidden.shape[0]

    # Remove illegal elements
    same_preds = get_positive_mask(preds)
    same_labels = get_positive_mask(labels)
    diff_labels = get_negative_mask(labels)

    same_preds_same_labels = torch.multiply(same_preds, same_labels)
    same_preds_diff_labels = torch.multiply(same_preds, diff_labels)

    pos_mask = get_positive_mask(labels)
    # neg_mask = tf.ones_like(pos_mask) - pos_mask # INCORRECT, because it consider the diagonal as NEGATIVE
    neg_mask = get_negative_mask(labels) # CORRECT
    hard_neg_mask = get_hard_negative_mask(labels, preds)
    soft_neg_mask = get_soft_negative_mask(labels, preds)
    merge_mask = torch.minimum(hard_neg_mask+soft_neg_mask, torch.ones_like(hard_neg_mask))

    # ATTENTION HERE, NEED VERIFIED BY EXPERIMENT, Stop_gradient make it worse
    # pos_mask = pos_mask.detach()
    # neg_mask = neg_mask.detach()
    # hard_neg_mask = hard_neg_mask.detach()
    # soft_neg_mask = soft_neg_mask.detach()
    # merge_mask = merge_mask.detach()
    # same_preds_same_labels = same_preds_same_labels.detach()
    #
    if neg_type == 'all':
        illegal_mask = torch.zeros_like(neg_mask)
        nb_neg = torch.sum(neg_mask, dim=-1)

    elif neg_type == 'hard':
        illegal_mask = torch.maximum(neg_mask - hard_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(hard_neg_mask, dim=-1)

    elif neg_type == 'soft':
        illegal_mask = torch.maximum(neg_mask - soft_neg_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(soft_neg_mask, dim=-1)

    elif neg_type == 'merge':
        illegal_mask = torch.maximum(neg_mask - merge_mask, torch.zeros_like(neg_mask))
        nb_neg = torch.sum(merge_mask, dim=-1)

    elif neg_type == 'leaking':
        illegal_mask = torch.maximum(torch.ones_like(same_preds_same_labels) - same_preds_same_labels - same_preds_diff_labels, torch.zeros_like(same_preds_same_labels))
        #print(illegal_mask)
        nb_neg = torch.sum(same_preds_diff_labels, dim=-1)

    # Pairwise Similarity
    masks = F.one_hot(torch.arange(batch_size), num_classes=batch_size).to(hidden.device)
    sim = - distance(hidden, hidden, dist, pairwise=True) / temperature
    logits_12 = sim - masks * LARGE_NUM
    logits_masked = logits_12 - LARGE_NUM * illegal_mask
    logprobs = F.log_softmax(logits_masked, dim=1)

    """ Convert labels to pairwise labels of shape [2b, 2b], diagonal(labels)=0
        multiple hot label vectors

    """
    # Cross entropy loss with multiple hot labels

    if neg_type == 'leaking':
        loss = - torch.sum(same_preds_same_labels * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(same_preds_same_labels, dim=-1), torch.ones_like(loss)) # [2b, ]
    else:
        loss = - torch.sum(pos_mask * logprobs, dim=-1) # [2b, ]
        nb_pos = torch.maximum(torch.sum(pos_mask, dim=-1), torch.ones_like(loss)) # [2b, ]

    # if True:
    #     scale = torch.div(nb_neg, nb_neg+nb_pos) # V1
    #     # scale = torch.div(nb_neg, nb_pos) #V3
    #     # scale = nb_neg #V2
    #     loss = torch.multiply(loss, scale)

    assert(loss.shape == nb_pos.shape)
    #print(nb_pos)
    loss = torch.div(loss, nb_pos) # [2b,]
    loss = torch.mean(loss, dim=0) # [1,]

    return loss

In [ ]:
# Define parameters for the loss function
dist = 'cosine'
neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
hidden_norm = False
temperature = 0.9

# Compute the loss
loss = soft_lcscl(z_proj, labels_one_hot, preds, dist, neg_type, hidden_norm, temperature)
print("Computed Loss:", loss.item())

In [ ]:
import torch
import torch.nn.functional as F

# Define a small batch size and dimension
batch_size = 2
dim = 3

# Hidden representations (4, 3) because we have 2*batch_size examples
hidden = torch.tensor([
    [0.2, 0.8, 0.1],
    [0.4, 0.5, 0.7],
    [0.1, 0.2, 0.9],
    [0.6, 0.3, 0.1]
], dtype=torch.float32)

# Labels (4, num_classes), here num_classes is 2 for simplicity
labels = torch.tensor([
    [1, 0],  # Example 1: Class 0
    [1, 0],  # Example 2: Class 0
    [0, 1],  # Example 3: Class 1
    [0, 1]   # Example 4: Class 1
], dtype=torch.float32)

# Predictions (4, num_classes), here num_classes is 2 for simplicity
preds = torch.tensor([
    [1, 0],  # Predicted Class 0
    [0, 1],  # Predicted Class 1
    [1, 0],  # Predicted Class 0
    [0, 1]   # Predicted Class 1
], dtype=torch.float32)


# Define parameters for the loss function
dist = 'cosine'
neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
hidden_norm = False
temperature = 0.07

# Compute the loss
loss = soft_lcscl(hidden, labels, preds, dist, neg_type, hidden_norm, temperature)
print("Computed Loss:", loss.item())

In [ ]:
def adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine):
    lr = learning_rate
    if cosine:
        eta_min = lr * (lr_decay_rate ** 3)
        lr = eta_min + (lr - eta_min) * (1 + math.cos(math.pi * epoch / epochs)) / 2
    else:
        steps = np.sum(epoch > np.asarray([int(0.7*epochs), int(0.8*epochs), int(0.9*epochs)]))  # Update as per your schedule
        if steps > 0:
            lr = lr * (lr_decay_rate ** steps)

    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

def warmup_learning_rate(batch_size, epoch, batch_id, total_batches, optimizer, cosine, learning_rate, lr_decay_rate, warm_up=False):
    if (warm_up) or (batch_size > 256):
        warm_epochs = 10
        warmup_from = (0.1 * learning_rate)

        if epoch <= warm_epochs:
            p = (batch_id + (epoch - 1) * total_batches) / (warm_epochs * total_batches)
            if cosine:
                eta_min = learning_rate * (lr_decay_rate ** 3)
                warmup_to = eta_min + (learning_rate - eta_min) * (1 + math.cos(math.pi * warm_epochs / epochs)) / 2
            else:
                warmup_to = learning_rate

            lr = warmup_from + p * (warmup_to - warmup_from)
            for param_group in optimizer.param_groups:
                param_group['lr'] = lr

def save_model(model, optimizer, epoch, save_file):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
    }, save_file)

In [ ]:
def pgd_min2(x, y, model, insertion_array, removal_array, k=25, step_length=0.02, norm='linf',rounding=False, initial_rounding_threshold=0.5, round_threshold=0.5, random=False, is_report_loss_diff=True, is_sample=False):
    """
    Projected Gradient Descent (PGD) adversarial attack (loss based on goal's class, which we have to minimize the loss).
    :param y: Ground truth labels
    :param x: Feature vector
    :param model: Neural network model
    :param k: Number of steps
    :param step_length: Step size for each iteration
    :param norm: Norm used for perturbation ('linf' or 'l2')
    :param initial_rounding_threshold: Threshold parameter for rounding the initial x_next
    :param round_threshold: Threshold parameter for rounding
    :param random: Flag to generate random thresholds
    :param is_report_loss_diff: Flag to report loss difference
    :param is_sample: Flag to sample randomly from the feasible area
    :return: The adversarial version of x (tensor)
    """
    model.eval()

    # Compute natural loss
    criterion = nn.CrossEntropyLoss(reduction='none')
    loss_natural = criterion(model(x), torch.zeros_like(y.view(-1).long()))

    # Initialize starting point
    x_next = x.clone()
    x_next = get_x0(x_next, initial_rounding_threshold, is_sample)


    # Expand insertion_array and removal_array to match the batch size
    expanded_insertion_array = insertion_array.expand(x.shape[0], -1)
    expanded_removal_array = removal_array.expand(x.shape[0], -1)

    # Update insertion and removal arrays based on input x
    insertion_array_updated = torch.bitwise_or(expanded_insertion_array, x.to(torch.uint8))
    removal_array_updated = torch.bitwise_or(expanded_removal_array, 1 - x.to(torch.uint8))

    loss_steps_i = []
    loss_steps_d = []

    # Multi-step PGD
    for t in range(k):
        #print('*********** ',t)
        # Forward pass
        x_var = x_next.clone().detach().requires_grad_(True)
        y_model = model(x_var)
        loss = criterion(y_model, torch.zeros_like(y.view(-1).long()))
        #print(loss)
        #print('loss_mal : ',criterion(y_model, torch.zeros_like(y.view(-1).long())))
        loss_steps_i.append(criterion(y_model, y.view(-1).long()).mean().detach().item())
        loss_steps_d.append(loss.mean().detach().item())

        # Compute gradient
        grad_vars = torch.autograd.grad(loss.mean(), x_var)
        gradients = -(grad_vars[0].data)
        #print('torch.abs(gradients).sum() : ',torch.abs(gradients).sum(dim=-1))

        #grad4insertion = (gradients >= 0) *(x_var < 1.)* insertion_array_updated * gradients
        grad4insertion = (gradients >= 0) *(x_var < 1.)*  gradients
        grad4removal = (gradients < 0) * (x_var > 0.) * removal_array_updated * gradients

        gradients = grad4removal + grad4insertion
        #print(torch.abs(gradients).sum())

        # Norm
        if norm == 'linf':
            perturbation = torch.sign(gradients).float()

        elif norm == 'free':
            perturbation = gradients

        elif norm == 'max':
            max_values, _ = torch.max(torch.abs(gradients), dim=-1, keepdim=True)
            perturbation = (gradients / (max_values+ 1e-20))

        elif norm == 'l2':
            l2norm = torch.linalg.norm(gradients, dim=-1, keepdim=True)
            perturbation = (gradients / (l2norm + 1e-20)).float()

        elif norm == 'l1':
            # consider just features of a sample which are not updated yet(because our update is 0to1 or 1to0 not stepwise)
            un_mod = torch.abs(x - x_var) <= 1e-6
            gradients = gradients * un_mod
            max_grad = torch.topk(torch.abs(gradients).view(gradients.size(0), -1), 1, dim=-1)[0]
            #print('max_grad ',max_grad)
            perturbation = (torch.abs(gradients) >= max_grad).float() * torch.sign(gradients).float()
            done = get_done(x_next, y, model)
            if torch.all(done):
                break
            perturbation[done] = 0.

        else:
            raise ValueError("Expect 'l1' or 'l2' or 'linf' norm.")

        # Update x_next
        x_next = torch.clamp(x_next + perturbation * step_length, min=0., max=1.)

    # Rounding step
    #if random:
       #round_threshold = torch.rand(x_next.size())
    if rounding:
        x_next = round_x(x_next, round_threshold=round_threshold)

    # Compute adversarial loss
    loss_adv = criterion(model(x_next), torch.zeros_like(y.view(-1).long())).data

    # Replace with natural if adversarial loss is higher
    replace_flag = (loss_adv > loss_natural).squeeze()
    x_next[replace_flag] = x[replace_flag]

    diff = torch.abs(x_next - x).sum(dim=-1)

    if is_report_loss_diff:
        done = get_done(x_next, y, model)
        print(f"PGD {norm}: Attack effectiveness {done.sum().item() / x.size()[0] * 100:.3f}%.")
        print('mean of differnce featres',torch.mean(diff))

    return x_next


def get_loss(x,y,model):
    criterion = nn.CrossEntropyLoss(reduction='none')
    outputs = model(x)
    loss = criterion(outputs, y.view(-1).long())
    _, predicted = torch.topk(outputs, k=1)
    done = (predicted != y).squeeze()

    return loss, done

def get_done(x, y, model):
    # Get the model's predictions
    outputs = model(x)

    # Use argmax to get the predicted class indices
    predicted = torch.argmax(outputs, dim=1)

    # Ensure y is in the same shape as predicted for comparison
    y = y.view_as(predicted)

    # Determine if the predictions are incorrect
    done = (predicted != y).bool()

    return done

def PGD_Max2(x,y, model, insertion_array, removal_array, attack_list = ['linf', 'l2', 'l1'],k=10,steps_max=5, rounding=False, is_sample = False, varepsilon = 1e-20, is_report_loss_diff = False):
    """
    PGD_Max adversarial attack.

    Args:
        x: Input data tensor (shape: [samples, features])
        y: Ground truth labels tensor (shape: [samples])
        model: Neural network model
        attack_list: List of norms for attacks (default: ['linf', 'l2', 'l1'])
        steps_max: Maximum number of steps (default: 5)
        is_sample: Flag to sample randomly from the feasible area (default: False)
        vaρεpsilon: Tolerance for stopping condition (default: 1e-20)

    Returns:
        Adversarial version of input data (tensor)
    """
    batch_size = x.shape[0]
    norm_params = {
        'l1': {'k': k, 'step_length': 1.0,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff},
        'l2': {'k': k, 'step_length': 0.5,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff},
        'linf': {'k': k, 'step_length': 0.02,'rounding':rounding, 'is_report_loss_diff':is_report_loss_diff}
    }

    model.eval()
    with torch.no_grad():
        loss, done = get_loss(x,y,model) #shape:[samples],[samples]

    pre_loss = loss
    n = x.shape[0]
    adv_x = x.detach().clone().to(x.device)
    stop_flag = torch.zeros(n, dtype=torch.bool).to(x.device) #[samples]

    for t in range(steps_max):
      num_remaining  = (~stop_flag).sum().item()
      #print('number of remaining samples : ',num_remaining )
      if num_remaining  <= 0:
          break

      remaining_label = y[~stop_flag]
      pertbx = []

      for norm in attack_list:
          if norm in norm_params:
              params = norm_params[norm]
              perturbation = pgd_min2(adv_x[~stop_flag], remaining_label, model, insertion_array, removal_array, norm=norm, is_sample=is_sample, **params)
              #print("the number of added features : ", (perturbation.sum() - adv_x[~stop_flag].sum())/len(adv_x[~stop_flag]))
              pertbx.append(perturbation)
          else:
              raise ValueError("Expected 'l1' or 'l2' or 'linf' norm.")


      # here pertbx.shape = a list of (number of attacks  ,(num_remaining ,features))
      pertbx = torch.vstack(pertbx)
      # here pertbx.shape = a tensor (num_remaining *number of attacks samples, features)

      with torch.no_grad():
        remaining_label_ext = torch.cat([remaining_label] * len(attack_list)) #(labels*number of attacks )
        loss, done = get_loss(pertbx, remaining_label_ext,model) #(labels*number of attacks )

        # for a sample, if there is at least one successful attack, we will select the one with maximum loss;
        # while if no attacks evade the victim successful, all perturbed examples are reminded for selection
        max_v = loss.amax()
        loss[done] += max_v

        loss = loss.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)
        done = done.reshape(len(attack_list), num_remaining ).permute(1, 0) #(num_remaining ,number of attacks)

        success_flag = torch.any(done, dim=-1) #(num_remaining )

        pertbx = pertbx.reshape(len(attack_list), num_remaining , x.shape[1]).permute([1, 0, 2])#(num_remaining ,attacks,features)
        _, indices = loss.max(dim=-1) # ans:(samples), max loss among attacks which worked, and max loss among all attacks for sample , none of them worked
        adv_x[~stop_flag] = pertbx[torch.arange(num_remaining ), indices]
        a_loss = loss[torch.arange(num_remaining ), indices]
        pre_stop_flag = stop_flag.clone()
        stop_flag[~stop_flag] = (torch.abs(pre_loss[~stop_flag] - a_loss) < varepsilon) | success_flag
        pre_loss[~pre_stop_flag] = a_loss

    return adv_x

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# Define the Linear Classifier(NO NEED)
class LinearClassifier(nn.Module):
    def __init__(self, dim_in=200, num_classes=2):
        super(LinearClassifier, self).__init__()
        self.fc = nn.Linear(dim_in, num_classes)

    def forward(self, x):
        return self.fc(x)


# Define the CoreModel
class CoreModel(nn.Module):
    def __init__(self):
        super(CoreModel, self).__init__()
        self.fc1 = nn.Linear(10000, 200)
        self.fc2 = nn.Linear(200, 200)
        self.dropout2 = nn.Dropout(0.6)
        self.fc3 = nn.Linear(200, 2)

    def forward(self, x, return_z=False):
        x = F.relu(self.fc1(x))
        z = F.relu(self.fc2(x))
        z2 = self.dropout2(z)
        output  = self.fc3(z2)
        if return_z:
            return output, z

        return output

# Define the Projection Head
class ProjHead(nn.Module):
    def __init__(self, dim_in=200, feat_dim=128):
        super(ProjHead, self).__init__()
        self.head = nn.Linear(dim_in, feat_dim)

    def forward(self, x):
        x = F.normalize(x, dim=1)
        z_proj = self.head(x)
        return z_proj



# Define the Wrapper
class Wrapper(nn.Module):
    def __init__(self, CoreModel, ProjLayer):
        super(Wrapper, self).__init__()
        self.CoreModel = CoreModel
        self.ProjLayer = ProjLayer
        #self.LC = LC

    def forward(self, x, return_z=False):
        output, z = self.CoreModel(x, return_z=True)
        z_proj = self.ProjLayer(z)
        #output = self.LC(z)
        if return_z:
            return output, z_proj
        else:
            return output

# Instantiate the components
core_model = CoreModel()
proj_head = ProjHead(200, 128)
#linear_classifier = LinearClassifier(128, 2)

# Combine them in the Wrapper
combined_model = Wrapper(core_model, proj_head)

# Example usage
x = torch.randn(2, 10000)  # Example input
output, z_proj = combined_model(x,return_z=True)
print(output.shape)
print(z_proj.shape)

In [ ]:
def adversarial_training2(model, train_loader, val_loader, attack, adv_epochs=100, lr=0.001, weight_decay=0., beta=1., device=device, verbose=True, **kwargs):
    # beta: Float, penalty factor for adversarial loss
    # Assuming positive class (malware) is label 1
    #class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class

    cosine = True
    lr_decay_rate = 0.1

    # Define Loss Function and Optimizer
    #criterion = nn.CrossEntropyLoss(weight=class_weights)
    criterion_at = nn.CrossEntropyLoss()
    criterion_scl = SupConLoss(temperature=0.07)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_time = 0.
    nbatches = len(train_loader)
    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    for epoch in range(adv_epochs):
        epoch_losses = []
        epoch_accuracies = []

        adjust_learning_rate(optimizer, epoch+1, adv_epochs, lr, lr_decay_rate, cosine)

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            if batch_size < 100:
                continue

            # Warm-up learning rate
            warmup_learning_rate(batch_size, epoch+1, idx_batch, len(train_loader), optimizer, cosine, lr, lr_decay_rate, warm_up=False)

            # Separate malicious and benign samples
            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            # Generate adversarial examples
            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
            y_batch = torch.cat([y_batch, mal_y_batch])

            model.train()

            # Forward pass and backward pass
            optimizer.zero_grad()
            outputs, z_proj = model(x_batch,return_z=True)
            # loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            loss_scl = criterion_scl(z_proj.unsqueeze(1), y_batch)
            loss_train = loss_at + beta * loss_scl

            #print(loss_train.item())
            loss_train.backward()
            optimizer.step()

            # Calculate metrics
            epoch_losses.append(loss_train.item())
            predicted = outputs.argmax(1).unsqueeze(1)
            acc_train = (predicted == y_batch).sum().item() / len(y_batch)
            epoch_accuracies.append(acc_train)

            # Print batch level information
            #if verbose:
                #print(f'Mini batch: {idx_batch + 1}/{nbatches} | Epoch: {epoch + 1}/{adv_epochs} | Batch Loss: {loss_train.item():.4f} | Batch Accuracy: {acc_train * 100:.2f}%')

        # Calculate epoch level metrics
        mean_loss = np.mean(epoch_losses)
        mean_accuracy = np.mean(epoch_accuracies) * 100

        # Print epoch level information
        if verbose:
            print(f'Epoch: {epoch+1}/{adv_epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        # Evaluation on validation set
        model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            # Generate adversarial examples for validation set
            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        # Calculate validation accuracy
        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        # Update best validation accuracy
        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        # Print validation results
        if verbose:
            print(f"\tVal accuracy(without attack) {(avg_acc_val) * 100:.4}% and accuracy(with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel select at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
def adversarial_training3(model, train_loader, val_loader, attack, adv_epochs=100, lr=0.001, weight_decay=0., beta=1., device=device, verbose=True, **kwargs):
    # beta: Float, penalty factor for adversarial loss
    # Assuming positive class (malware) is label 1
    #class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class

    cosine = True
    lr_decay_rate = 0.1

    # Define Loss Function and Optimizer
    #criterion = nn.CrossEntropyLoss(weight=class_weights)
    criterion_at = nn.CrossEntropyLoss()
    criterion_scl = SupConLoss(temperature=0.07)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_time = 0.
    nbatches = len(train_loader)
    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    for epoch in range(adv_epochs):
        epoch_losses = []
        epoch_accuracies = []

        adjust_learning_rate(optimizer, epoch+1, adv_epochs, lr, lr_decay_rate, cosine)

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            if batch_size < 200:
                continue

            # Warm-up learning rate
            warmup_learning_rate(batch_size, epoch+1, idx_batch, len(train_loader), optimizer, cosine, lr, lr_decay_rate, warm_up=False)

            # Separate malicious and benign samples
            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            # Randomly select 10% of malware samples
            num_malware = len(mal_y_batch)
            num_adv = int(num_malware * 0.1)
            selected_indices = np.random.choice(num_malware, num_adv, replace=False)

            # (first approach:substitution)
            # Generate adversarial examples
            #if len(selected_indices) > 0:
                #model.eval()
                #pertb_mal_x = attack(mal_x_batch[selected_indices], mal_y_batch[selected_indices], model, **kwargs)
                #pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)
                #mal_x_batch[selected_indices] = pertb_mal_x

            # (second approach:addition)
            # Generate adversarial examples
            if len(selected_indices) > 0:
                model.eval()
                pertb_mal_x = attack(mal_x_batch[selected_indices], mal_y_batch[selected_indices], model, **kwargs)
                #pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)
                x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
                y_batch = torch.cat([y_batch, mal_y_batch[selected_indices]])

            model.train()

            # Forward pass and backward pass
            optimizer.zero_grad()
            outputs, z_proj = model(x_batch,return_z=True)
            # loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())
            loss_scl = criterion_scl(z_proj.unsqueeze(1), y_batch)
            loss_train = loss_at + beta * loss_scl

            #print(loss_train.item())
            loss_train.backward()
            optimizer.step()

            # Calculate metrics
            epoch_losses.append(loss_train.item())
            predicted = outputs.argmax(1).unsqueeze(1)
            acc_train = (predicted == y_batch).sum().item() / len(y_batch)
            epoch_accuracies.append(acc_train)

            # Print batch level information
            #if verbose:
                #print(f'Mini batch: {idx_batch + 1}/{nbatches} | Epoch: {epoch + 1}/{adv_epochs} | Batch Loss: {loss_train.item():.4f} | Batch Accuracy: {acc_train * 100:.2f}%')

        # Calculate epoch level metrics
        mean_loss = np.mean(epoch_losses)
        mean_accuracy = np.mean(epoch_accuracies) * 100

        # Print epoch level information
        if verbose:
            print(f'Epoch: {epoch+1}/{adv_epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        # Evaluation on validation set
        model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            # Generate adversarial examples for validation set
            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        # Calculate validation accuracy
        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        # Update best validation accuracy
        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        # Print validation results
        if verbose:
            print(f"\tVal accuracy(without attack) {(avg_acc_val) * 100:.4}% and accuracy(with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel select at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
def adversarial_training4(model, train_loader, val_loader, attack, adv_epochs=100, lr=0.001, weight_decay=0., beta=1., device=device, verbose=True, **kwargs):
    # beta: Float, penalty factor for adversarial loss
    # Assuming positive class (malware) is label 1
    #class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class

    cosine = True
    lr_decay_rate = 0.1

    # Define parameters for the loss function
    dist = 'cosine'
    neg_type = 'leaking'  # You can change this to 'soft', 'merge', or 'leaking'
    hidden_norm = False
    temperature = 0.07

    # Define Loss Function and Optimizer
    #criterion = nn.CrossEntropyLoss(weight=class_weights)
    criterion_at = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_time = 0.
    nbatches = len(train_loader)
    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    for epoch in range(adv_epochs):
        epoch_losses = []
        epoch_accuracies = []

        adjust_learning_rate(optimizer, epoch+1, adv_epochs, lr, lr_decay_rate, cosine)

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            if batch_size < 100:
                continue

            # Warm-up learning rate
            warmup_learning_rate(batch_size, epoch+1, idx_batch, len(train_loader), optimizer, cosine, lr, lr_decay_rate, warm_up=False)

            # Separate malicious and benign samples
            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            # Generate adversarial examples
            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
            y_batch = torch.cat([y_batch, mal_y_batch])

            model.train()

            # Forward pass and backward pass
            optimizer.zero_grad()
            outputs, z_proj = model(x_batch,return_z=True)
            predicted = outputs.argmax(1).unsqueeze(1)
            # loss
            loss_at = criterion_at(outputs, y_batch.view(-1).long())

            labels_one_hot = F.one_hot(y_batch.long(), num_classes=2).squeeze(1).float()
            predicted_one_hot = F.one_hot(predicted.long(), num_classes=2).squeeze(1).float()
            loss_scl = soft_lcscl(z_proj, labels_one_hot, predicted_one_hot, dist, neg_type, hidden_norm, temperature)
            #loss_scl = criterion_scl(z_proj.unsqueeze(1), y_batch)
            loss_train = loss_at + beta * loss_scl

            #print(loss_train.item())
            loss_train.backward()
            optimizer.step()

            # Calculate metrics
            epoch_losses.append(loss_train.item())
            #predicted = outputs.argmax(1).unsqueeze(1)
            acc_train = (predicted == y_batch).sum().item() / len(y_batch)
            epoch_accuracies.append(acc_train)

            # Print batch level information
            #if verbose:
                #print(f'Mini batch: {idx_batch + 1}/{nbatches} | Epoch: {epoch + 1}/{adv_epochs} | Batch Loss: {loss_train.item():.4f} | Batch Accuracy: {acc_train * 100:.2f}%')

        # Calculate epoch level metrics
        mean_loss = np.mean(epoch_losses)
        mean_accuracy = np.mean(epoch_accuracies) * 100

        # Print epoch level information
        if verbose:
            print(f'Epoch: {epoch+1}/{adv_epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        # Evaluation on validation set
        model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            # Generate adversarial examples for validation set
            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        # Calculate validation accuracy
        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        # Update best validation accuracy
        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        # Print validation results
        if verbose:
            print(f"\tVal accuracy(without attack) {(avg_acc_val) * 100:.4}% and accuracy(with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel select at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
class SupConLoss(nn.Module):
    """Supervised Contrastive Learning: https://arxiv.org/pdf/2004.11362.pdf.
    It also supports the unsupervised contrastive loss in SimCLR"""
    def __init__(self, temperature=0.07, contrast_mode='all',
                 base_temperature=0.07):
        super(SupConLoss, self).__init__()
        self.temperature = temperature
        self.contrast_mode = contrast_mode
        self.base_temperature = base_temperature

    def forward(self, features, labels=None, mask=None):
        """Compute loss for model. If both `labels` and `mask` are None,
        it degenerates to SimCLR unsupervised loss:
        https://arxiv.org/pdf/2002.05709.pdf

        Args:
            features: hidden vector of shape [bsz, n_views, ...].
            labels: ground truth of shape [bsz].
            mask: contrastive mask of shape [bsz, bsz], mask_{i,j}=1 if sample j
                has the same class as sample i. Can be asymmetric.
        Returns:
            A loss scalar.
        """
        device = (torch.device('cuda')
                  if features.is_cuda
                  else torch.device('cpu'))

        if len(features.shape) < 3:
            raise ValueError('`features` needs to be [bsz, n_views, ...],'
                             'at least 3 dimensions are required')
        if len(features.shape) > 3:
            features = features.view(features.shape[0], features.shape[1], -1)

        batch_size = features.shape[0]
        if labels is not None and mask is not None:
            raise ValueError('Cannot define both `labels` and `mask`')
        elif labels is None and mask is None:
            mask = torch.eye(batch_size, dtype=torch.float32).to(device)
        elif labels is not None:
            labels = labels.contiguous().view(-1, 1)
            if labels.shape[0] != batch_size:
                raise ValueError('Num of labels does not match num of features')
            mask = torch.eq(labels, labels.T).float().to(device)
        else:
            mask = mask.float().to(device)

        contrast_count = features.shape[1]
        contrast_feature = torch.cat(torch.unbind(features, dim=1), dim=0)
        if self.contrast_mode == 'one':
            anchor_feature = features[:, 0]
            anchor_count = 1
        elif self.contrast_mode == 'all':
            anchor_feature = contrast_feature
            anchor_count = contrast_count
        else:
            raise ValueError('Unknown mode: {}'.format(self.contrast_mode))

        # compute logits
        anchor_dot_contrast = torch.div(
            torch.matmul(anchor_feature, contrast_feature.T),
            self.temperature)
        # for numerical stability
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        # tile mask
        mask = mask.repeat(anchor_count, contrast_count)
        # mask-out self-contrast cases
        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size * anchor_count).view(-1, 1).to(device),
            0
        )
        mask = mask * logits_mask

        # compute log_prob
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True))

        # compute mean of log-likelihood over positive
        # modified to handle edge cases when there is no positive pair
        # for an anchor point.
        # Edge case e.g.:-
        # features of shape: [4,1,...]
        # labels:            [0,1,1,2]
        # loss before mean:  [nan, ..., ..., nan]
        mask_pos_pairs = mask.sum(1)
        mask_pos_pairs = torch.where(mask_pos_pairs < 1e-6, 1, mask_pos_pairs)
        mean_log_prob_pos = (mask * log_prob).sum(1) / mask_pos_pairs

        # loss
        loss = - (self.temperature / self.base_temperature) * mean_log_prob_pos
        loss = loss.view(anchor_count, batch_size).mean()

        return loss

In [ ]:
def adversarial_training(model, criterion_at, criterion_scl, optimizer, train_loader, attack, epochs, learning_rate, beta=1.,
                         cosine=True, lr_decay_rate=0.1, device=device, verbose=True, **kwargs):
    # beta: Float, penalty factor for adversarial loss
    # Assuming positive class (malware) is label 1

    nbatches = len(train_loader)

    for epoch in range(epochs):

        adjust_learning_rate(optimizer, epoch, epochs, learning_rate, lr_decay_rate, cosine)

        epoch_losses = []
        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            if batch_size < 100:
                continue

            # Warm-up learning rate
            warmup_learning_rate(batch_size, epoch, idx_batch, len(train_loader), optimizer, cosine, learning_rate, lr_decay_rate, warm_up=False)

            # Separate malicious and benign samples
            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]




            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            #pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)
            x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
            y_batch = torch.cat([y_batch, mal_y_batch])


            model.train()

            # Forward pass and backward pass
            optimizer.zero_grad()
            output, z_proj = model(x_batch,return_z=True)
            # loss
            loss_at = criterion_at(output, y_batch.view(-1).long())
            loss_scl = criterion_scl(z_proj.unsqueeze(1), y_batch)
            loss_train = loss_at + beta * loss_scl

            #print(loss_train.item())
            loss_train.backward()
            optimizer.step()

            # Calculate metrics
            epoch_losses.append(loss_train.item())

            # Print batch level information
            #if verbose:
                #print(f'Mini batch: {idx_batch + 1}/{nbatches} | Epoch: {epoch + 1}/{epochs} | Batch Loss: {loss_train.item():.4f} | Batch Accuracy: {acc_train * 100:.2f}%')

        # Calculate epoch level metrics
        mean_loss = np.mean(epoch_losses)

        # Print epoch level information
        if verbose:
            print(f'Epoch: {epoch+1}/{epochs} | Training loss (epoch level): {mean_loss:.4f} ')

In [ ]:
def adversarial_training(model, criterion, optimizer, train_loader, attack, adv_epochs, learning_rate, beta=0.05, device=device, verbose=True, **kwargs):
    # beta: Float, penalty factor for adversarial loss
    # Assuming positive class (malware) is label 1

    total_time = 0.
    nbatches = len(train_loader)
    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    for epoch in range(adv_epochs):
        epoch_losses = []
        epoch_accuracies = []

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):

            if len(y_batch) < 500:
                continue

            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            # Separate malicious and benign samples
            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            # Randomly select 10% of malware samples
            num_malware = len(mal_y_batch)
            num_adv = int(num_malware * beta)
            selected_indices = np.random.choice(num_malware, num_adv, replace=False)

            # (first approach:substitution)
            # Generate adversarial examples
            #if len(selected_indices) > 0:
                #model.eval()
                #pertb_mal_x = attack(mal_x_batch[selected_indices], mal_y_batch[selected_indices], model, **kwargs)
                #pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)
                #mal_x_batch[selected_indices] = pertb_mal_x

            # (second approach:addition)
            # Generate adversarial examples
            if len(selected_indices) > 0:
                model.eval()
                pertb_mal_x = attack(mal_x_batch[selected_indices], mal_y_batch[selected_indices], model, **kwargs)
                pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)
                x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
                y_batch = torch.cat([y_batch, mal_y_batch[selected_indices]])


            model.train()

            # Forward pass and backward pass
            optimizer.zero_grad()
            features = model(x_batch)
            # loss
            loss_train = criterion(features.unsqueeze(1), y_batch.view(-1).long())

            #print(loss_train.item())
            loss_train.backward()
            optimizer.step()

            # Calculate metrics
            epoch_losses.append(loss_train.item())

            # Print batch level information
            #if verbose:
                #print(f'Mini batch: {idx_batch + 1}/{nbatches} | Epoch: {epoch + 1}/{adv_epochs} | Batch Loss: {loss_train.item():.4f} | Batch Accuracy: {acc_train * 100:.2f}%')

        # Calculate epoch level metrics
        mean_loss = np.mean(epoch_losses)

        # Print epoch level information
        if verbose:
            print(f'Epoch: {epoch+1}/{adv_epochs} | Training loss (epoch level): {mean_loss:.4f} ')

In [ ]:
def adversarial_training3(model, train_loader, val_loader, attack, adv_epochs=50, lr=0.001, weight_decay=0., beta=0.01, device=device, verbose=True, **kwargs):
    # beta: Float, penalty factor for adversarial loss
    # Assuming positive class (malware) is label 1
    class_weights = torch.tensor([0.11, 0.89]).to(device)  # Adjust the weights based on the class distribution, higher weight for positive class

    # Define Loss Function and Optimizer
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    #criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    total_time = 0.
    nbatches = len(train_loader)
    best_acc_val = 0.
    acc_val_adv_be = 0.
    best_epoch = 0

    for epoch in range(adv_epochs):
        epoch_losses = []
        epoch_accuracies = []

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):
            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            # Separate malicious and benign samples
            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            # Generate adversarial examples
            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)
            x_batch = torch.cat([x_batch, pertb_mal_x], dim=0)
            y_batch = torch.cat([y_batch, mal_y_batch])
            model.train()

            # Forward pass and backward pass
            optimizer.zero_grad()
            outputs = model(x_batch)
            # loss of normal samples
            loss_train = criterion(outputs[:batch_size], y_batch[:batch_size].view(-1).long())
            #print(loss_train.item())
            #loss of adversarial examples
            if len(mal_y_batch) > 0:
              loss_train += beta * criterion(outputs[batch_size:], y_batch[batch_size:].view(-1).long())
            else:
              print('no malware in this batch')
            #print(loss_train.item())
            loss_train.backward()
            optimizer.step()

            # Calculate metrics
            epoch_losses.append(loss_train.item())
            predicted = outputs.argmax(1).unsqueeze(1)
            acc_train = (predicted == y_batch).sum().item() / len(y_batch)
            epoch_accuracies.append(acc_train)

            # Print batch level information
            #if verbose:
                #print(f'Mini batch: {idx_batch + 1}/{nbatches} | Epoch: {epoch + 1}/{adv_epochs} | Batch Loss: {loss_train.item():.4f} | Batch Accuracy: {acc_train * 100:.2f}%')

        # Calculate epoch level metrics
        mean_loss = np.mean(epoch_losses)
        mean_accuracy = np.mean(epoch_accuracies) * 100

        # Print epoch level information
        if verbose:
            print(f'Epoch: {epoch+1}/{adv_epochs} | Training loss (epoch level): {mean_loss:.4f} | Train accuracy: {mean_accuracy:.2f}%')

        # Evaluation on validation set
        model.eval()
        cor_val = 0
        cor_ad_val = 0
        n_samples = 0
        n_ad_samples = 0
        for x_val, y_val in val_loader:
            x_val, y_val = x_val.to(torch.float32).to(device), y_val.to(device)
            n_samples += len(x_val)
            outputs = model(x_val)
            predicted = outputs.argmax(1).unsqueeze(1)
            cor_val += (predicted == y_val).sum().item()

            # Generate adversarial examples for validation set
            mal_x_batch, mal_y_batch = x_val[y_val.squeeze() == 1], y_val[y_val.squeeze() == 1]
            n_ad_samples += len(mal_x_batch)
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            outputs = model(pertb_mal_x)
            y_pred = outputs.argmax(1)
            cor_ad_val += (y_pred == 1.).sum().item()

        # Calculate validation accuracy
        assert n_ad_samples > 0
        avg_acc_val = (cor_val / n_samples)
        avg_acc_ad_val = (cor_ad_val / n_ad_samples)
        acc_all = (avg_acc_val + avg_acc_ad_val) / 2.

        # Update best validation accuracy
        if acc_all >= best_acc_val:
            best_acc_val = acc_all
            acc_val_adv_be = avg_acc_ad_val
            best_epoch = epoch + 1
            torch.save(model.state_dict(), 'model_AT_max.pth')

        # Print validation results
        if verbose:
            print(f"\tVal accuracy(without attack) {(avg_acc_val) * 100:.4}% and accuracy(with attack) {(avg_acc_ad_val) * 100:.4}% under attack and overall accuracy {acc_all * 100:.4}%.")
            print(f"\tModel select at epoch {best_epoch} with validation accuracy {best_acc_val * 100:.4}% and accuracy {acc_val_adv_be * 100:.4}% under attack.")

In [ ]:
# Define the Encoder Network
class Encoder(nn.Module):
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6):
        super(Encoder, self).__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim1)
        self.fc2 = nn.Linear(hidden_dim1, hidden_dim2)
        #self.dropout = nn.Dropout(dropout_prob)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        #x = self.dropout(x)
        x = self.fc2(x)
        out = self.relu(x)
        return out

class SupConNet(nn.Module):
    """backbone + projection head"""
    def __init__(self, input_dim=10000, hidden_dim1=200, hidden_dim2=200, dropout_prob=0.6, feat_dim=64, head='mlp'):
        super(SupConNet, self).__init__()
        self.encoder = Encoder(input_dim, hidden_dim1, hidden_dim2, dropout_prob)
        if head == 'linear':
            self.head = nn.Linear(hidden_dim2, feat_dim)
        elif head == 'mlp':
            self.head = nn.Sequential(
                nn.Linear(hidden_dim2, feat_dim),
                nn.ReLU(inplace=True),
                nn.Linear(feat_dim, 2)
            )
        else:
            raise NotImplementedError(
                'head not supported: {}'.format(head))

    def forward(self, x):
        feat = self.encoder(x)
        feat = F.normalize(self.head(feat), dim=1)
        return feat

class CombinedModel(nn.Module):
    def __init__(self, encoder, classifier):
        super(CombinedModel, self).__init__()
        self.encoder = encoder
        self.classifier = classifier

    def forward(self, x):
        #with torch.no_grad():  # Ensure encoder weights are not updated
        features = self.encoder(x)
        output = self.classifier(features)
        return output

## Step 3 — Adversarial training round 1, then the attack battery

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 1024
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)
combined_model = Wrapper(core_model, proj_head).to(device)

epochs = 100

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':10, 'rounding':False,'steps_max': 5, 'is_report_loss_diff' : False}

adversarial_training4(combined_model.to(device), train_loader, val_loader, PGD_Max2, adv_epochs=100, lr=0.02, weight_decay=0., beta=1., device=device, verbose=True, **attack_params)

In [ ]:
combined_model2 = Wrapper(core_model, proj_head).to(device)
combined_model2.load_state_dict(torch.load('/content/model_AT_max.pth'))

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 100, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':100,'steps_max': 5, 'rounding':True, 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2.to(device), PGD_Max2, device, **attack_params)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":100, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":100, 'is_report_loss_diff' : False}),
    (bca_k, {"k":100, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":100, 'is_report_loss_diff' : False}),
    #(mimicry, {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':1,'is_sample':False, 'varepsilon':1e-20}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':5,'is_sample':False, 'varepsilon':1e-20}),
    (StepwiseMax_onestep2, {'insertion_array':insertion_array, 'removal_array':removal_array,'step_lengths':{"l1": 1.0, "l2": 0.05, "linf": 0.0008}, "steps":650}),

]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model2, attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

In [ ]:
features, labels = extract_features(combined_model2, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

## Step 4 — Mimicry against the trained model

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":30, 'is_report_loss_diff':False}
adv_predict(test_loader, combined_model, mimicry, device, **attack_params)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}
adv_predict(test_loader, combined_model, mimicry, device, **attack_params)

## Step 5 — Rounds 2 and 3, with evaluations

In [ ]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)
combined_model = Wrapper(core_model, proj_head).to(device)

epochs = 100

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':20, 'rounding':False,'steps_max': 5, 'is_report_loss_diff' : False}

adversarial_training4(combined_model.to(device), train_loader, val_loader, PGD_Max2, adv_epochs=100, lr=0.02, weight_decay=0., beta=1., device=device, verbose=True, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 250, 'step_length': 0.5, 'norm': 'l2', 'rounding':True, 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min2, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'k':250,'steps_max': 5, 'rounding':True, 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model.to(device), PGD_Max2, device, **attack_params)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":100, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":100, 'is_report_loss_diff' : False}),
    (bca_k, {"k":100, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":100, 'is_report_loss_diff' : False}),
    #(mimicry, {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':1,'is_sample':False, 'varepsilon':1e-20}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':5,'is_sample':False, 'varepsilon':1e-20}),
    (StepwiseMax_onestep2, {'insertion_array':insertion_array, 'removal_array':removal_array,'step_lengths':{"l1": 1.0, "l2": 0.05, "linf": 0.0008}, "steps":650}),

]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model.to(device), attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

In [ ]:
features, labels = extract_features(combined_model, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

In [ ]:
model_evaluation(combined_model2.to(device), test_loader, device)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model2, pgd_min2, device, **attack_params)

In [ ]:
# Define different attacks with their parameters
attacks = [
    (dfgsm_k, {"k":100, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}),
    (bga_k, {"k":100, 'is_report_loss_diff' : False}),
    (bca_k, {"k":100, 'is_report_loss_diff' : False}),
    (grosse_k, {"k":100, 'is_report_loss_diff' : False}),
    (mimicry, {'removal_array':removal_array, "trials":100, 'is_report_loss_diff':False}),
    (pgd_min2, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 1., 'norm': 'l1', 'is_report_loss_diff' : False}),
    (pgd_min2, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}),
    (pgd_min2, {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':1,'is_sample':False, 'varepsilon':1e-20}),
    (PGD_Max2, {'insertion_array':insertion_array, 'removal_array':removal_array,'steps_max':5,'is_sample':False, 'varepsilon':1e-20}),
    (StepwiseMax_onestep2, {'insertion_array':insertion_array, 'removal_array':removal_array,'step_lengths':{"l1": 1.0, "l2": 0.05, "linf": 0.0008}, "steps":650}),

]

# Iterate over each attack and its parameters
for attack_func, attack_params in attacks:
    print(f"Running attack: {attack_func.__name__} with parameters: {attack_params}")
    adv_predict(test_loader, combined_model2, attack_func, device, **attack_params)
    print('********************************************')  # Print an empty line for separation

In [ ]:
# Define the DataLoader for training, validation, and test sets
batch_size = 256
#sampler = weight_sampler(y_train)
#train_loader = DataLoader(train_dataset, batch_size=batch_size,sampler=sampler)
train_loader = DataLoader(train_dataset, batch_size=batch_size,shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)
combined_model = Wrapper(core_model, proj_head).to(device)

epochs = 100

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 10, 'step_length': 0.5, 'norm': 'l2', 'rounding':False, 'is_report_loss_diff' : False}

adversarial_training2(combined_model.to(device), train_loader, val_loader, pgd_min2, adv_epochs=100, lr=0.001, weight_decay=0., beta=1., device=device, verbose=True, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 250, 'step_length': 0.5, 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min2, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model.to(device), PGD_Max2, device, **attack_params)

## Step 6 — The joint objective: adversarial loss + SupCon loss

In [ ]:
import torch.optim as optim

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model, loss function, and optimizer
core_model = CoreModel().to(device)
proj_head = ProjHead(200, 128).to(device)
combined_model = Wrapper(core_model, proj_head).to(device)

learning_rate = 0.001
momentum = 0.9
weight_decay = 5e-4
cosine = True

criterion_at = nn.CrossEntropyLoss()
criterion_scl = SupConLoss(temperature=0.07)
optimizer = optim.Adam(combined_model.parameters(), learning_rate)

epochs = 100

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 50, 'step_length': 0.1, 'norm': 'l2', 'is_report_loss_diff' : False}

adversarial_training(combined_model.to(device), criterion_at, criterion_scl, optimizer, train_loader, pgd_min2, epochs, learning_rate, beta=1., device=device, verbose=True, **attack_params)

In [ ]:
# 500
features, labels = extract_features(combined_model, test_loader)
tsne_results = apply_tsne(features)
plot_tsne_and_density(tsne_results, labels)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 250, 'step_length': 0.02, 'norm': 'linf', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min2, device, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'k': 250, 'step_length': 1., 'norm': 'l2', 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model, pgd_min2, device, **attack_params)

In [ ]:
attack_params =  {"k":200, "epsilon":0.02, 'random':True, 'is_report_loss_diff' : False}
adv_predict(test_loader, combined_model.to(device), dfgsm_k, device, **attack_params)

## Step 7 — End-to-end: SupConNet trained adversarially

# New Section

In [ ]:
model_contrast = SupConNet().to(device)
model_contrast

In [ ]:
model_contrast = SupConNet().to(device)
criterion = SupConLoss(temperature=0.07)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.05
momentum = 0.9
cosine = True
#optimizer =  optim.SGD(model_contrast.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
optimizer = optim.Adam(model_contrast.parameters(), lr=learning_rate,weight_decay=weight_decay)
adv_epochs = 200

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}


adversarial_training3(model_contrast.to(device), train_loader, val_loader, PGD_Max2, adv_epochs=200, lr=0.0005, weight_decay=0., beta=0.01, device=device, verbose=True, **attack_params)

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}
adv_predict(test_loader, model_contrast.to(device), PGD_Max2, device, **attack_params)

In [ ]:
attack_params =  {'removal_array':removal_array, "trials":10, 'is_report_loss_diff':False}
adv_predict(test_loader, model_contrast, mimicry, device, **attack_params)

In [ ]:
# Extract features from validation set
features, labels = extract_features(model_contrast, val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

In [ ]:
SupConNet(feat_dim=128, head='linear')

In [ ]:
model_contrast = SupConNet(feat_dim=128, head='linear').to(device)
criterion = SupConLoss(temperature=0.07)
weight_decay = 1e-4
lr_decay_rate = 0.1
learning_rate = 0.005
momentum = 0.9
cosine = True
#optimizer =  optim.SGD(model_contrast.parameters(),lr=learning_rate,momentum=momentum,weight_decay=weight_decay)
optimizer = optim.Adam(model_contrast.parameters(), lr=learning_rate,weight_decay=weight_decay)
adv_epochs = 200

In [ ]:
attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}

adversarial_training2(model_contrast.to(device), criterion, optimizer, train_loader, PGD_Max2, adv_epochs, learning_rate, beta=0.02, device=device, verbose=True, **attack_params)

In [ ]:
# Extract features from validation set
features, labels = extract_features(model_contrast.encoder, val_loader)

# Apply t-SNE
tsne_results = apply_tsne(features)

# Plot joint plot
plot_tsne_and_density(tsne_results, labels)

## Step 8 — Fine-tuning the pretrained encoder, then the final battery

In [ ]:
def adversarial_training2(model, criterion, optimizer, train_loader, attack, adv_epochs, learning_rate, beta=0.01, device=device, verbose=True, **kwargs):
    # beta: Float, penalty factor for adversarial loss
    # Assuming positive class (malware) is label 1

    total_time = 0.
    nbatches = len(train_loader)

    for epoch in range(adv_epochs):
        epoch_losses = []
        epoch_accuracies = []

        for idx_batch, (x_batch, y_batch) in enumerate(train_loader):

            if len(y_batch) < 500:
                continue

            x_batch, y_batch = x_batch.to(torch.float32).to(device), y_batch.to(device)
            batch_size = x_batch.shape[0]

            # Separate malicious and benign samples
            mal_x_batch, ben_x_batch = x_batch[y_batch.squeeze() == 1], x_batch[y_batch.squeeze() == 0]
            mal_y_batch, ben_y_batch = y_batch[y_batch.squeeze() == 1], y_batch[y_batch.squeeze() == 0]

            # Generate adversarial examples
            model.eval()
            pertb_mal_x = attack(mal_x_batch, mal_y_batch, model, **kwargs)
            pertb_mal_x = round_x(pertb_mal_x, round_threshold=0.5)

            x_batch2 = torch.cat([ben_x_batch, pertb_mal_x], dim=0)
            y_batch2 = torch.cat([ben_y_batch, mal_y_batch])


            model.train()

            # Forward pass and backward pass
            optimizer.zero_grad()

            # loss of normal samples
            outputs = model(x_batch)
            loss_train = criterion(outputs, y_batch.view(-1).long())
            #print(loss_train.item())
            #loss of adversarial examples
            if len(mal_y_batch) > 0:
              outputs2 = model(x_batch2)
              loss_train += beta * criterion(outputs2, y_batch2.view(-1).long())
            else:
              print('no malware in this batch')

            #print(loss_train.item())
            loss_train.backward()
            optimizer.step()

            # Calculate metrics
            epoch_losses.append(loss_train.item())

            # Print batch level information
            #if verbose:
                #print(f'Mini batch: {idx_batch + 1}/{nbatches} | Epoch: {epoch + 1}/{adv_epochs} | Batch Loss: {loss_train.item():.4f} | Batch Accuracy: {acc_train * 100:.2f}%')

        # Calculate epoch level metrics
        mean_loss = np.mean(epoch_losses)

        # Print epoch level information
        if verbose:
            print(f'Epoch: {epoch+1}/{adv_epochs} | Training loss (epoch level): {mean_loss:.4f} ')

In [ ]:
# Extract the pretrained encoder
pretrained_encoder = model_contrast.encoder

# Set the parameters of the encoder to not require gradients
for param in pretrained_encoder.parameters():
    param.requires_grad = False

In [ ]:
classifier = LinearClassifier().to(device)
combined_model = CombinedModel(pretrained_encoder, classifier).to(device)

class_weights = torch.tensor([0.11, 0.89]).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(combined_model.classifier.parameters(), lr=learning_rate)

learning_rate = 0.005
cosine = True
adv_epochs = 200

attack_params =  {'insertion_array':insertion_array, 'removal_array':removal_array,'attack_list':['linf', 'l2', 'l1'],'steps_max': 5, 'is_report_loss_diff' : False}
adversarial_training2(combined_model.to(device), criterion, optimizer, train_loader, PGD_Max2, adv_epochs, learning_rate, beta=0.02, device=device, verbose=True, **attack_params)